# 📊 Evaluasi Akurasi Jumlah Deteksi Pelat Nomor YOLO (`best.pt`) vs Ground Truth
Notebook ini mengevaluasi **khusus jumlah pelat nomor yang terdeteksi oleh model `models/best.pt`** dibandingkan dengan jumlah *Ground Truth* pada **seluruh gambar di dataset** (`train`, `val`, `test`).

Output yang dihasilkan:
1. **Daftar seluruh gambar** beserta jumlah pelat *Ground Truth*, jumlah terdeteksi YOLO (`best.pt`), selisih, dan status kecocokannya.
2. **Ringkasan akurasi per split** (`train`, `val`, `test`).
3. **Akurasi Total** keseluruhan dataset (Exact Match Accuracy & Rasio Jumlah Pelat).

---
## 1. Setup Environment & Model `models/best.pt`

In [1]:
import os
import sys
import glob
import subprocess

for mod_name, pip_name in [
    ("kagglehub", "kagglehub"),
    ("ultralytics", "ultralytics"),
    ("pandas", "pandas"),
]:
    try:
        __import__(mod_name)
    except ImportError:
        print(f"[Setup] Menginstal {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

REPO_URL = "https://github.com/richieese-lab/Indonesian-License-Plate-Detection-Recognition.git"
REPO_BRANCH = "main"
COLAB_REPO_DIR = "/content/Indonesian-License-Plate-Detection-Recognition"

project_root = os.getcwd()
if not os.path.exists(os.path.join(project_root, "models", "best.pt")):
    if not os.path.exists(COLAB_REPO_DIR):
        print("🌐 Menarik repository & models/best.pt dari GitHub ke Colab...")
        try:
            subprocess.check_call(["git", "clone", "-b", REPO_BRANCH, REPO_URL, COLAB_REPO_DIR])
        except subprocess.CalledProcessError:
            subprocess.check_call(["git", "clone", "-b", "Richard", REPO_URL, COLAB_REPO_DIR])
    os.chdir(COLAB_REPO_DIR)
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.insert(0, project_root)

BEST_PT_PATH = os.path.join(project_root, "models", "best.pt")
if not os.path.exists(BEST_PT_PATH):
    raise FileNotFoundError(f"File models/best.pt tidak ditemukan di {BEST_PT_PATH}")

print(f"✅ Direktori kerja : {project_root}")
print(f"✅ Model YOLO      : {BEST_PT_PATH}")

✅ Direktori kerja : /content/Indonesian-License-Plate-Detection-Recognition
✅ Model YOLO      : /content/Indonesian-License-Plate-Detection-Recognition/models/best.pt


---
## 2. Akses Seluruh Dataset via Kaggle API (`kagglehub`) / Fallback Lokal

In [2]:
dataset_root = None
try:
    import kagglehub
    print("🌐 Mengakses dataset via Kaggle API (kagglehub)...")
    dataset_root = kagglehub.dataset_download("juanthomaswijaya/indonesian-license-plate-dataset")
    print("📁 Path dataset:", dataset_root)
except Exception as e:
    print(f"⚠️ Kaggle API dilewati ({e}). Mencari di direktori lokal...")

if not dataset_root or not os.path.exists(dataset_root):
    for cand in [
        os.path.join(project_root, "Dataset", "images"),
        os.path.join(project_root, "Dataset", "Raw"),
        os.path.join(project_root, "Dataset"),
    ]:
        if os.path.exists(cand):
            dataset_root = cand
            break

# Temukan folder induk 'Indonesian License Plate Dataset' (bukan Recognition)
plate_ds_dir = None
for root_dir, dirs, _ in os.walk(dataset_root):
    if "Recognition" in root_dir:
        continue
    if "images" in dirs and ("labels" in dirs or "labelswithLP" in dirs):
        plate_ds_dir = root_dir
        break

if not plate_ds_dir:
    raise FileNotFoundError("Folder dataset deteksi plat (images + labels) tidak ditemukan.")

label_folder_name = "labels" if os.path.exists(os.path.join(plate_ds_dir, "labels")) else "labelswithLP"
print(f"✅ Dataset Deteksi Plat : {plate_ds_dir}")
print(f"✅ Folder Ground Truth  : {label_folder_name}")

🌐 Mengakses dataset via Kaggle API (kagglehub)...
Using Colab cache for faster access to the 'indonesian-license-plate-dataset' dataset.
📁 Path dataset: /kaggle/input/indonesian-license-plate-dataset
✅ Dataset Deteksi Plat : /kaggle/input/indonesian-license-plate-dataset/Indonesian License Plate Dataset
✅ Folder Ground Truth  : labels


---
## 3. Jalankan Evaluasi Jumlah Pelat (`best.pt` vs Ground Truth) untuk Seluruh Gambar

In [3]:
import pandas as pd
from ultralytics import YOLO

# Pengaturan evaluasi
CONF_THRESHOLD = 0.35
SPLITS_TO_EVAL = ["train", "val", "test"]

model = YOLO(BEST_PT_PATH)
rows = []

for split in SPLITS_TO_EVAL:
    img_dir = os.path.join(plate_ds_dir, "images", split)
    lbl_dir = os.path.join(plate_ds_dir, label_folder_name, split)
    if not os.path.exists(img_dir):
        continue

    img_files = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    print(f"⏳ Memproses split '{split}' ({len(img_files)} gambar)...")

    for fname in img_files:
        img_path = os.path.join(img_dir, fname)
        stem = os.path.splitext(fname)[0]
        lbl_path = os.path.join(lbl_dir, f"{stem}.txt")

        # Hitung Ground Truth
        gt_count = 0
        if os.path.exists(lbl_path):
            with open(lbl_path, "r", encoding="utf-8") as f:
                gt_count = sum(1 for line in f if line.strip())

        # Hitung deteksi YOLO (best.pt)
        res = model(img_path, conf=CONF_THRESHOLD, verbose=False)
        pred_count = sum(len(r.boxes) for r in res)

        is_exact = (pred_count == gt_count)
        if gt_count == 0 and pred_count == 0:
            img_acc = 100.0
        elif max(gt_count, pred_count) == 0:
            img_acc = 0.0
        else:
            img_acc = round((min(gt_count, pred_count) / max(gt_count, pred_count)) * 100.0, 2)

        if is_exact:
            status = "Cocok"
        elif pred_count < gt_count:
            status = f"Kurang ({pred_count - gt_count})"
        else:
            status = f"Lebih (+{pred_count - gt_count})"

        rows.append({
            "Split": split,
            "Gambar": fname,
            "GT_Jumlah": gt_count,
            "YOLO_Terdeteksi": pred_count,
            "Selisih": pred_count - gt_count,
            "Cocok_Tepat": is_exact,
            "Akurasi_Gambar (%)": img_acc,
            "Status": status,
        })

df = pd.DataFrame(rows)
print(f"✅ Evaluasi selesai! Total gambar diproses: {len(df)}")

⏳ Memproses split 'train' (800 gambar)...
⏳ Memproses split 'val' (100 gambar)...
⏳ Memproses split 'test' (100 gambar)...
✅ Evaluasi selesai! Total gambar diproses: 1000


---
## 4. Ringkasan Akurasi Total & Per Split

In [4]:
total_images = len(df)
exact_matches = int(df["Cocok_Tepat"].sum())
total_gt_plates = int(df["GT_Jumlah"].sum())
total_yolo_plates = int(df["YOLO_Terdeteksi"].sum())

exact_match_acc = (exact_matches / total_images * 100.0) if total_images else 0.0
mean_img_acc = float(df["Akurasi_Gambar (%)"].mean()) if total_images else 0.0
mae_count = float(df["Selisih"].abs().mean()) if total_images else 0.0

print("=" * 72)
print("📈 RINGKASAN AKURASI DETEKSI JUMLAH PELAT (YOLO best.pt vs GROUND TRUTH)")
print("=" * 72)
print(f"• Confidence Threshold YOLO      : {CONF_THRESHOLD}")
print(f"• Total Gambar Dievaluasi        : {total_images} gambar")
print(f"• Total Pelat Ground Truth       : {total_gt_plates} pelat")
print(f"• Total Pelat Terdeteksi (best.pt): {total_yolo_plates} pelat")
print(f"• Gambar dengan Jumlah Cocok Tepat: {exact_matches} / {total_images} gambar")
print(f"• AKURASI TOTAL (Exact Match)    : {exact_match_acc:.2f}%")
print(f"• AKURASI TOTAL (Rata-rata Rasio): {mean_img_acc:.2f}%")
print(f"• Rata-rata Error Jumlah (MAE)   : {mae_count:.4f} pelat/gambar")
print("=" * 72)

# Ringkasan per Split
split_summary = df.groupby("Split", sort=False).agg(
    Total_Gambar=("Gambar", "count"),
    Total_GT=("GT_Jumlah", "sum"),
    Total_YOLO=("YOLO_Terdeteksi", "sum"),
    Jumlah_Cocok=("Cocok_Tepat", "sum"),
    Akurasi_Exact_Match=("Cocok_Tepat", lambda s: round(s.mean() * 100.0, 2)),
    Akurasi_Rasio_Rata2=("Akurasi_Gambar (%)", lambda s: round(s.mean(), 2)),
).reset_index()
display(split_summary)

📈 RINGKASAN AKURASI DETEKSI JUMLAH PELAT (YOLO best.pt vs GROUND TRUTH)
• Confidence Threshold YOLO      : 0.35
• Total Gambar Dievaluasi        : 1000 gambar
• Total Pelat Ground Truth       : 1906 pelat
• Total Pelat Terdeteksi (best.pt): 1986 pelat
• Gambar dengan Jumlah Cocok Tepat: 822 / 1000 gambar
• AKURASI TOTAL (Exact Match)    : 82.20%
• AKURASI TOTAL (Rata-rata Rasio): 92.75%
• Rata-rata Error Jumlah (MAE)   : 0.1900 pelat/gambar


,Split,Total_Gambar,Total_GT,Total_YOLO,Jumlah_Cocok,Akurasi_Exact_Match,Akurasi_Rasio_Rata2
0,train,800,1530,1600,656,82.0,92.69
1,val,100,179,188,86,86.0,93.83
2,test,100,197,198,80,80.0,92.15


---
## 5. Daftar Lengkap Hasil Per Gambar (Seluruh Dataset)

In [5]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Simpan ke CSV agar mudah dibuka di Excel bila diperlukan
os.makedirs(os.path.join(project_root, "Output"), exist_ok=True)
csv_out = os.path.join(project_root, "Output", "evaluasi_jumlah_pelat_yolo.csv")
df.to_csv(csv_out, index=False)
print(f"💾 Daftar lengkap seluruh gambar juga disimpan ke: {csv_out}\n")

print("📋 DAFTAR SELURUH GAMBAR (GROUND TRUTH vs YOLO best.pt):")
display(df[["Split", "Gambar", "GT_Jumlah", "YOLO_Terdeteksi", "Selisih", "Akurasi_Gambar (%)", "Status"]])

💾 Daftar lengkap seluruh gambar juga disimpan ke: /content/Indonesian-License-Plate-Detection-Recognition/Output/evaluasi_jumlah_pelat_yolo.csv

📋 DAFTAR SELURUH GAMBAR (GROUND TRUTH vs YOLO best.pt):


,Split,Gambar,GT_Jumlah,YOLO_Terdeteksi,Selisih,Akurasi_Gambar (%),Status
0,train,train001.jpg,1,1,0,100.00,Cocok
1,train,train002.jpg,2,1,-1,50.00,Kurang (-1)
2,train,train003.jpg,2,2,0,100.00,Cocok
3,train,train004.jpg,2,1,-1,50.00,Kurang (-1)
4,train,train005.jpg,2,2,0,100.00,Cocok
5,train,train006.jpg,1,3,2,33.33,Lebih (+2)
6,train,train007.jpg,1,1,0,100.00,Cocok
7,train,train008.jpg,2,2,0,100.00,Cocok
8,train,train009.jpg,1,1,0,100.00,Cocok
9,train,train010.jpg,1,1,0,100.00,Cocok


---
## 6. Daftar Khusus Gambar yang Jumlah Deteksinya Tidak Cocok

In [6]:
df_mismatch = df[~df["Cocok_Tepat"]].reset_index(drop=True)
print(f"⚠️ Total gambar dengan jumlah deteksi tidak cocok: {len(df_mismatch)} dari {len(df)} gambar")
if len(df_mismatch) > 0:
    display(df_mismatch[["Split", "Gambar", "GT_Jumlah", "YOLO_Terdeteksi", "Selisih", "Akurasi_Gambar (%)", "Status"]])
else:
    print("🎉 Semua gambar cocok 100%!")

⚠️ Total gambar dengan jumlah deteksi tidak cocok: 178 dari 1000 gambar


,Split,Gambar,GT_Jumlah,YOLO_Terdeteksi,Selisih,Akurasi_Gambar (%),Status
0,train,train002.jpg,2,1,-1,50.00,Kurang (-1)
1,train,train004.jpg,2,1,-1,50.00,Kurang (-1)
2,train,train006.jpg,1,3,2,33.33,Lebih (+2)
3,train,train013.jpg,1,2,1,50.00,Lebih (+1)
4,train,train026.jpg,2,3,1,66.67,Lebih (+1)
5,train,train043.jpg,1,2,1,50.00,Lebih (+1)
6,train,train047.jpg,1,2,1,50.00,Lebih (+1)
7,train,train055.jpg,1,2,1,50.00,Lebih (+1)
8,train,train073.jpg,1,0,-1,0.00,Kurang (-1)
9,train,train076.jpg,2,4,2,50.00,Lebih (+2)
